## GSAT trend patterns

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os

import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprocess

In [ ]:
import src.slurm_cluster as scluster
client, scluster = scluster.init_dask_slurm_cluster()

In [ ]:
# dir_in ='/work/mh0033/m301036/Land_surf_temp/Disentangling_OBS_SAT_trend/Supp_Figure6_Forced/data/'
# MMEM_annual_ensemble_mean = xr.open_mfdataset(dir_in + 'MMEM_annual_ano_ensemble_mean_1950_2022.nc',chunks={'lat': 10, 'lon': 10})

In [ ]:
# Input the MMEM annual mean SAT data
input_model = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/LE_data/'

CanESM_data   = xr.open_mfdataset(input_model + 'tas_CanESM5_annual_ano_1850_2022.nc',chunks = {'run':1})
CESM2_data    = xr.open_mfdataset(input_model + 'SMBB_tas_annual_mean_all_members.nc',chunks = {'member':1}).rename({'member':'run'})
IPSL_data     = xr.open_mfdataset(input_model + 'tas_IPSL_CM6A_annual_ano_1850_2022.nc',chunks = {'run':1})
EC_Earth_data = xr.open_mfdataset(input_model + 'tas_EC_Earth3_annual_ano_1850_2022.nc',chunks = {'run':1})
ACCESS_data   = xr.open_mfdataset(input_model + 'tas_ACCESS_annual_ano_1850_2022.nc',chunks = {'run':1})
MPI_ESM_data  = xr.open_mfdataset(input_model + 'tas_MPI_ESM_annual_ano_1850_2022.nc',chunks = {'run':1})
MIROC_data    = xr.open_mfdataset(input_model + 'tas_MIROC6_annual_ano_1850_2022.nc',chunks = {'run':1})
# MMEM_annual_data = xr.open_mfdataset(input_model + 'tas_MMEM_annual_anomalies_ds.nc',chunks = {'lat':10,'lon':10})

In [ ]:
# Extreact each SMILEs 1950-2022 data then output the ensemble mean anomalies
CanESM_ano = CanESM_data.tas.sel(year=slice('1950','2022')).squeeze()
CESM2_ano = CESM2_data.TREFHT.sel(year=slice('1950','2022')).squeeze()
IPSL_ano = IPSL_data.tas.sel(year=slice('1950','2022')).squeeze()
EC_Earth_ano = EC_Earth_data.tas.sel(year=slice('1950','2022')).squeeze()
ACCESS_ano = ACCESS_data.tas.sel(year=slice('1950','2022')).squeeze()
MPI_ESM_ano = MPI_ESM_data.tas.sel(year=slice('1950','2022')).squeeze()
MIROC_ano = MIROC_data.tas.sel(year=slice('1950','2022')).squeeze()

In [ ]:
CanESM_ano_mean = CanESM_ano.mean(dim='run')
CESM2_ano_mean = CESM2_ano.mean(dim='run')
IPSL_ano_mean = IPSL_ano.mean(dim='run')
EC_Earth_ano_mean = EC_Earth_ano.mean(dim='run')
ACCESS_ano_mean = ACCESS_ano.mean(dim='run')
MPI_ESM_ano_mean = MPI_ESM_ano.mean(dim='run')
MIROC_ano_mean = MIROC_ano.mean(dim='run')

In [ ]:
CESM2_ano_mean

In [ ]:
dir_output ='/work/mh0033/m301036/OBS_LPS_revision/docs/data/LE_data/ENS/'
# os.makedirs(dir_output, exist_ok=True)
# CanESM_ano_mean.to_netcdf(dir_output + 'CanESM5_annual_ano_ensemble_mean_1950_2022.nc')
# CESM2_ano_mean.to_netcdf(dir_output + 'CESM2_annual_ano_ensemble_mean_1950_2022_cmip6+smbb.nc')
# IPSL_ano_mean.to_netcdf(dir_output + 'IPSL_CM6A_annual_ano_ensemble_mean_1950_2022.nc')
# EC_Earth_ano_mean.to_netcdf(dir_output + 'EC_Earth3_annual_ano_ensemble_mean_1950_2022.nc')
# ACCESS_ano_mean.to_netcdf(dir_output + 'ACCESS_annual_ano_ensemble_mean_1950_2022.nc')
# MPI_ESM_ano_mean.to_netcdf(dir_output + 'MPI_ESM_annual_ano_ensemble_mean_1950_2022.nc')
# MIROC_ano_mean.to_netcdf(dir_output + 'MIROC6_annual_ano_ensemble_mean_1950_2022.nc')

In [ ]:
CESM2_ano_mean

In [ ]:
# Concatenate the data
ds_list = [
	CanESM_ano_mean,
	CESM2_ano_mean,
	IPSL_ano_mean,
	EC_Earth_ano_mean,
	ACCESS_ano_mean,
	MPI_ESM_ano_mean,
	MIROC_ano_mean,
]

# Drop optional height coord if present to ensure consistent concatenation
ds_list = [ds.drop_vars("height", errors="ignore") for ds in ds_list]

MMEM_annual_data = xr.concat(ds_list, dim="run", coords="minimal", compat="override")

In [ ]:
MMEM_annual_ensemble_mean = MMEM_annual_data.sel(year=slice(1950,2022)).mean(dim='run')

In [ ]:
MMEM_annual_ensemble_mean

In [ ]:
# MMEM_annual_ensemble_mean.to_netcdf(dir_output + 'MMEM_annual_ano_ensemble_mean_1950_2022_smbbCESM2_complement.nc')

### Calculate the 1950-20222 Model-ENS simulated trend patterns

In [ ]:
# separate the data into two sets: Pre-1950 and Post-1950
"""
    Calcualte the trend pattern for the forced variability on each grid point for the consective intervals starting from 
    1940-1949, 1935-1949, 1930-1949, 1925-1949, 1920-1949, 1915-1949, 1910-1949, 1905-1949, 1900-1949, 1895-1949, 1890-1949, 1885-1949, 
    1880-1949, 1875-1949, 1870-1949, 1865-1949, 1860-1949, 1855-1949, 1850-1949
    Put the data segemnet into the same dataset
"""
start_year = 1950
end_year   = 2022
min_length = 10

def func_mk(x):
    results = data_process.mk_test(x)
    slope, p_val = results[0], results[1]
    return slope, p_val

In [ ]:
def process_realization(forced):
    """
    Compute MK trends for a single realization.
    """
    forced = MMEM_annual_ensemble_mean

    # Ensure time dimension is 'year'
    if "year" not in forced.dims:
        if "time" in forced.dims:
            forced = forced.rename({"time": "year"})
        else:
            raise ValueError(f"MMEM_annual_ensemble_mean has no 'year' or 'time' dimension.")

    forced = forced.sel(year=slice(start_year, end_year))
    lat = forced["lat"]
    lon = forced["lon"]

    trend_list, pvalue_list, period_names = [], [], []

    for begin_year in range(start_year, end_year - min_length + 2):
        time_slice = forced.sel(year=slice(begin_year, end_year))

        trend, p_values = xr.apply_ufunc(
            func_mk,
            time_slice,
            input_core_dims=[["year"]],
            output_core_dims=[[], []],
            vectorize=True,
            dask="parallelized",
            output_dtypes=[float, float],
            dask_gufunc_kwargs={"allow_rechunk": True},
        )

        period_key = f"{begin_year}-{end_year}"
        period_names.append(period_key)
        trend_list.append(trend)
        pvalue_list.append(p_values)

    trend_all = xr.concat([t * 10.0 for t in trend_list], dim="period")
    pval_all  = xr.concat(pvalue_list, dim="period")

    trend_all = trend_all.assign_coords(
        period=("period", period_names),
        lat=lat,
        lon=lon,
    )
    pval_all = pval_all.assign_coords(
        period=("period", period_names),
        lat=lat,
        lon=lon,
    )

    trend_all.name = "trend"
    pval_all.name  = "p_value"

    ds_out = xr.Dataset({"trend": trend_all, "p_value": pval_all})
    return ds_out

In [ ]:
MMEM_annual_tas_trend_da = process_realization(MMEM_annual_ensemble_mean)
MMEM_annual_tas_trend_da

In [ ]:
output_model = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/MMLE/'
os.makedirs(output_model, exist_ok=True)
MMEM_annual_tas_trend_da.to_netcdf(output_model + 'MMLE_MK_trend_1950_2022_sliding.nc')

### Plotting with the Robinson Projections

In [ ]:
plt.rcParams['figure.figsize'] = (8, 10)
plt.rcParams['font.size'] = 16
# plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['ytick.direction'] = 'out'
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.major.right'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['xtick.bottom'] = True

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.colors import BoundaryNorm, ListedColormap

def plot_trend_with_significance(trend_data, lats, lons, p_values, GMST_p_values=None, levels=None, extend=None,cmap=None, 
                                 title="", ax=None, show_xticks=False, show_yticks=False):
    """
    Plot the trend spatial pattern using Robinson projection with significance overlaid.

    Parameters:
    - trend_data: 2D numpy array with the trend values.
    - lats, lons: 1D arrays of latitudes and longitudes.
    - p_values: 2D array with p-values for each grid point.
    - GMST_p_values: 2D array with GMST p-values for each grid point.
    - title: Title for the plot.
    - ax: Existing axis to plot on. If None, a new axis will be created.
    - show_xticks, show_yticks: Boolean flags to show x and y axis ticks.
    
    Returns:
    - contour_obj: The contour object from the plot.
    """

    # Create a new figure/axis if none is provided
    if ax is None:
        fig, ax = plt.subplots(figsize=(20, 15), subplot_kw={'projection': ccrs.Robinson()})
        ax.set_global()
  
    # Determine significance mask (where p-values are less than 0.05)
    insignificance_mask = p_values >= 0.05
    
    # Plotting
    # contour_obj = ax.pcolormesh(lons, lats, trend_data,  cmap='RdBu_r',vmin=-5.0, vmax=5.0, transform=ccrs.PlateCarree(central_longitude=180), shading='auto')
    contour_obj = ax.contourf(lons, lats, trend_data, levels=levels, extend=extend, cmap=cmap, transform=ccrs.PlateCarree(central_longitude=0))

    # Plot significance masks with different hatches
    ax.contourf(lons, lats, insignificance_mask, levels=[0,0.05, 1.0],hatches=[None,'///'], colors='none', transform=ccrs.PlateCarree())

    ax.coastlines(resolution='110m')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                      color='gray', alpha=0.35, linestyle='--')

    # Disable labels on the top and right of the plot
    gl.top_labels = False
    gl.right_labels = False

    # Enable labels on the bottom and left of the plot
    gl.bottom_labels = show_xticks
    gl.left_labels = show_yticks
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    gl.xlabel_style = {'size': 14}
    gl.ylabel_style = {'size': 14}
    
    if show_xticks:
        gl.bottom_labels = True
    if show_yticks:
        gl.left_labels = True
    
    ax.set_title(title, loc='center', fontsize=18, pad=5.0)

    return contour_obj


In [ ]:
# define an asymmetric colormap
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.colors import BoundaryNorm
import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

# intervals = [-0.2, -0.15, -0.1, -0.05, 0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.9, 1.1, 1.3]
intervals = [-0.1, -0.075, -0.05, -0.025, 0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5, 2.75]

# Normalizing the intervals to [0, 1]
min_interval = min(intervals)
max_interval = max(intervals)
normalized_intervals = [(val - min_interval) / (max_interval - min_interval) for val in intervals]

# cmap = mcolors.ListedColormap(palettable.scientific.diverging.Vik_20.mpl_colors)
cmap=mcolors.ListedColormap(palettable.cmocean.diverging.Balance_20.mpl_colors)

# Define the colors at each interval
colors = [(0.00784313725490196, 0.2, 0.4627450980392157, 1.0),
    (0.00784313725490196, 0.2, 0.4627450980392157, 1.0),
    (0.023529411764705882, 0.32941176470588235, 0.5450980392156862, 1.0),
    (0.023529411764705882, 0.32941176470588235, 0.5450980392156862, 1.0),
    (1.0, 1.0, 1.0, 1.0),
    (1.0, 0.9, 0.98, 1.0),
    (1.0, 0.8, 0.5, 1.0),
    (1.0, 0.803921568627451, 0.607843137254902, 1.0), 
    (1.0, 0.6000000000000001, 0.20000000000000018, 1.0),
    (1.0, 0.4039215686274509, 0.0, 1.0),
    (0.8999999999999999, 0.19999999999999996, 0.0, 1.0),
    (0.7470588235294118, 0.0, 0.0, 1.0), 
    (0.6000000000000001, 0.0, 0.0, 1.0),
    (0.44705882352941173, 0.0, 0.0, 1.0),
    (0.30000000000000004, 0.0, 0.0, 1.0),
    (0.14705882352941177, 0.0, 0.0, 1.0),
    (0.0, 0.0, 0.0, 1.0)]

# Creating a list of tuples with normalized positions and corresponding colors
color_list = list(zip(normalized_intervals, colors))

# Create the colormap
custom_cmap = LinearSegmentedColormap.from_list('my_custom_cmap', color_list)

# Create a normalization
norm = Normalize(vmin=min_interval, vmax=max_interval)

### Plot the Original, Forced, MMEM trend patterns

In [ ]:
# check the min and max values of the trend data
trend_min = MMEM_annual_tas_trend_da.trend.min().item()
trend_max = MMEM_annual_tas_trend_da.trend.max().item()
print(f"Trend min: {trend_min}, Trend max: {trend_max}")

In [ ]:
lat = MMEM_annual_tas_trend_da['trend']['lat'].values
lon = MMEM_annual_tas_trend_da['trend']['lon'].values

titles = ["2013-2022(10yr)", "2003-2022(20yr)", "1993-2022(30yr)", "1983-2022(40yr)", "1973-2022(50yr)", "1963-2022(60yr)", "1953-2022(70yr)"]
titles_left = ["a.", "b.", "c.", "d.", "e.", "f.", "g."]

import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

# Get the actual period names from the dataset
available_periods = MMEM_annual_tas_trend_da['period'].values.tolist()
# Select the last 7 periods (most recent)
periods = available_periods[-7:]

# Define the GridSpec
fig = plt.figure(figsize=(25, 15))
gs = gridspec.GridSpec(3, 3, height_ratios=[1, 1, 1], width_ratios=[1, 1, 1], wspace=0.01, hspace=0.01)

extend= 'max'
for j, period in enumerate(periods):
    # Define the axes
    ax = fig.add_subplot(gs[j], projection=ccrs.Robinson(180))
    is_left = (j % 3 == 0)
    is_bottom_row = j >= (len(periods)//3)*3 
    
    trend_data = MMEM_annual_tas_trend_da['trend'].sel(period=period)
    trend_with_cyclic, lon_with_cyclic = cutil.add_cyclic_point(trend_data, coord=lon)
    p_values = MMEM_annual_tas_trend_da['p_value'].sel(period=period)
    p_values_with_cyclic, lon_with_cyclic = cutil.add_cyclic_point(p_values, coord=lon)
    levels = np.arange(-0.6, 0.65, 0.05)
    contour_obj = plot_trend_with_significance(trend_with_cyclic, lat, lon_with_cyclic, p_values_with_cyclic, 
                    GMST_p_values=None, levels=levels,extend=extend, cmap='twilight_shifted',
                    # cmap=mcolors.ListedColormap(palettable.cmocean.diverging.Balance_20.mpl_colors),
                    title=titles[j], ax=ax, show_xticks = is_bottom_row, show_yticks = is_left)
    ax.text(-0.03, 1.05, titles_left[j], fontsize=22,weight='bold', ha='center', va='center', rotation='horizontal', transform=ax.transAxes)

# Add horizontal colorbars
cbar_ax = fig.add_axes([0.43, 0.2, 0.5, 0.025])
cbar = plt.colorbar(contour_obj, cax=cbar_ax, orientation='horizontal')
cbar.ax.tick_params(labelsize=18)
cbar.set_label('Annual SAT Trend (°C/decade)', fontsize=22)

plt.tight_layout()
fig.savefig('MMEM_simulated_trend_Pattern_variations_1950-2022.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
client.close()
scluster.close()